## Structured Output

#### Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the outpur can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

#### Pydantic models provide the richest feature set with field validation, and nested structures.

In [24]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1152c8b10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1154de5d0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [25]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title : str = Field(description="The title of the movie")
    year : int = Field(description="This year the movie was released")
    director : str = Field(description="The director of the movie")
    rating : float = Field(description="The movies rating out of 10")

In [26]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.3', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1152c8b10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1154de5d0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the mov

In [27]:
model.invoke("Provide the details of movie Dhurandhar")

AIMessage(content='**“Dhurandhar” – Film Overview**\n\n| Item | Details |\n|------|---------|\n| **Title** | **Dhurandhar** |\n| **Language** | Hindi |\n| **Country** | India |\n| **Genre** | Action‑Drama / Thriller |\n| **Release Date** | 23\u202fMay\u202f2024 (theatrical release in India) |\n| **Running Time** | 2\u202fhr\u202f12\u202fmin |\n| **Production Company** | **Vijay Studios** (in association with **Sanjay Productions**) |\n| **Distributor** | **Reliance Entertainment** |\n| **Budget** | Approx. **₹\u202f85\u202fcrore** (≈\u202f$10\u202fmillion USD) |\n| **Box‑Office Gross** | ₹\u202f210\u202fcrore (worldwide) – declared a **“Super‑Hit”** by trade analysts |\n\n---\n\n## Creative Team\n\n| Role | Name |\n|------|------|\n| **Director** | **Anand Rao** (known for “Rudra” and “Kavach”) |\n| **Screenplay / Story** | **Priyanka Sharma** |\n| **Dialogues** | **Vikram Singh** |\n| **Producer(s)** | **Rohit Mehta**, **Sanjay Patel** |\n| **Cinematography** | **Sanjay K. Verma** |\n

In [28]:
model_with_structure.invoke("Provide details of movie Dhurandhar")

Movie(title='Dhurandhar', year=2022, director='Mahesh Manjrekar', rating=7.2)

### Message output alongside parsed structure

In [29]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""
    title : str = Field(..., description="The title of the movie")
    year : int = Field(..., description="This year the movie was released")
    director : str = Field(..., description="The director of the movie")
    rating : float = Field(..., description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details of movie Dhurandhar")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details of movie "Dhurandhar". Likely wants title, director, rating, year. We can call function Movie with those parameters. Need to know details. Dhurandhar is a Marathi film? Let\'s recall: "Dhurandhar" is a 2022 Marathi film starring? Actually "Dhurandhar" is a 2022 Marathi comedy drama directed by Mahesh Manjrekar? Wait Mahesh Manjrekar directed "Dhurandhar"? I think "Dhurandhar" 2022 is directed by Mahesh Manjrekar starring? Actually the Marathi movie "Dhurandhar" (2022) starring Mahesh Manjrekar? Let\'s quickly search memory: The film "Dhurandhar" is a 2022 Marathi drama about a man who becomes a police officer? I recall "Dhurandhar" starring Mahesh Manjrekar, maybe directed by Mahesh Manjrekar? Might be directed by Mahesh Manjrekar and starring? Could be "Mohan Joshi"? Not sure. Another possibility: "Dhurandhar" is a 2023 Hindi film? Actually there is a Hindi movie "Dhurandhar" starring? Not recall

### Nested Structure



In [41]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in crore INR")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details of the movie Dhurandhar")
response

    

MovieDetails(title='Dhurandhar', year=2022, cast=[Actor(name='Prasad Oak', role='Dhananjay'), Actor(name='Amruta Khanvilkar', role='Madhuri'), Actor(name='Shivaji Satam', role='Inspector'), Actor(name='Mohan Joshi', role='Supporting')], genres=['Comedy', 'Drama'], budget=None)

## TypeDict

#### TypeDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation.

In [52]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title : Annotated[str, ..., "The title of the movie"]
    year : Annotated[int, ..., "This year the movie was released"]
    director : Annotated[str, ..., "The director of the movie"]
    rating : Annotated[float, ..., "The movies rating out of 10"]

model_with_typedDict = model.with_structured_output(MovieDict)
response = model_with_typedDict.invoke("Provide details of the movie RaOne")
response

{'director': 'Anubhav Sinha', 'rating': 6.5, 'title': 'Ra.One', 'year': 2011}

In [56]:
from typing_extensions import TypedDict, Annotated


class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in crore INR")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details of the movie Swadesh")
response

    

{'budget': 200000000,
 'cast': [{'name': 'Shah Rukh Khan', 'role': 'Mohan Bhargava'},
  {'name': 'Gayatri Joshi', 'role': 'Gita'},
  {'name': 'Kishori Ballal', 'role': 'Kaveri Amma'},
  {'name': 'Dilip Prabhavalkar', 'role': "Mohan's Father"},
  {'name': 'Mithun Chakraborty', 'role': 'Sanjay Patel'}],
 'genres': ['Drama', 'Social', 'Romance'],
 'title': 'Swades',
 'year': 2004}

In [58]:
model.profile


{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

## Data Classes

#### A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [59]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name : str =Field(description= "The name of the person")
    email : str = Field(description= "The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format= ContactInfo         # Auto-selects ProvideStrategy
)

result = agent.invoke({
    "messages": [{"role":"user", "content" : "Extract contact info from: Raj, Raj@example.com, (231) 3241-1332"}]
})

result


{'messages': [HumanMessage(content='Extract contact info from: Raj, Raj@example.com, (231) 3241-1332', additional_kwargs={}, response_metadata={}, id='77809917-e205-494a-891a-3677e13024ae'),
  AIMessage(content='{"name":"Raj","email":"Raj@example.com","phone":"(231) 3241-1332"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info: name, email, phone. Provide JSON according to ContactInfo schema: fields name, email, phone, all required. Use compact JSON formatting. Provide only the final JSON.\n\nWe need to parse the input: "Raj, Raj@example.com, (231) 3241-1332". Name: Raj. Email: Raj@example.com. Phone: (231) 3241-1332. Probably keep format as given. Provide JSON: {"name":"Raj","email":"Raj@example.com","phone":"(231) 3241-1332"}.\n\nCheck schema: required fields present. Use compact JSON, no whitespace except necessary. Output only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 179, 'prompt_tokens': 230, 'total_tokens': 409, 'completion_

In [60]:
result["structured_response"]

ContactInfo(name='Raj', email='Raj@example.com', phone='(231) 3241-1332')

In [61]:
## Tyepdict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name : str       #  The name of the person")
    email : str      #  The email address of the person"
    phone: str       #  The phone number of the person"

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    response_format= ContactInfo         # Auto-selects ProvideStrategy
)

result = agent.invoke({
    "messages": [{"role":"user", "content" : "Extract contact info from: Raj, Raj@example.com, (231) 3241-1332"}]
})

result


{'messages': [HumanMessage(content='Extract contact info from: Raj, Raj@example.com, (231) 3241-1332', additional_kwargs={}, response_metadata={}, id='7f573713-9ada-4e83-ad3b-22a97b026a61'),
  AIMessage(content='{"name":"Raj","email":"Raj@example.com","phone":"(231) 3241-1332"}', additional_kwargs={'reasoning_content': 'The user wants to extract contact info from a string. The system says we must output only the final JSON object, adhering to the ContactInfo schema. Must include fields name, email, phone. Provide compact JSON.\n\nExtract: Name: Raj, Email: Raj@example.com, Phone: (231) 3241-1332. Probably keep as given.\n\nWe need to output JSON: {"name":"Raj","email":"Raj@example.com","phone":"(231) 3241-1332"}\n\nCheck schema: required fields present. Good.\n\nNow output only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 148, 'prompt_tokens': 212, 'total_tokens': 360, 'completion_time': 0.314640704, 'completion_tokens_details': {'reasoning_tokens': 116}, 'prompt_ti

In [62]:
result["structured_response"]

{'name': 'Raj', 'email': 'Raj@example.com', 'phone': '(231) 3241-1332'}

#### Data Class


In [63]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str    # The name of the person
    email: str   # The email address of the person
    phone: str   # The phone number of the person

agent = create_agent(
model="groq:openai/gpt-oss-120b",
response_format= ContactInfo         # Auto-selects ProvideStrategy
)

result = agent.invoke({
    "messages": [{"role":"user", "content" : "Extract contact info from: Raj, Raj@example.com, (231) 3241-1332"}]
})

result["structured_response"]

ContactInfo(name='Raj', email='Raj@example.com', phone='(231) 3241-1332')